In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/spect-heart-dataset/spect_train_binary.csv
/kaggle/input/spect-heart-dataset/spect_test_binary.csv
/kaggle/input/spect-heart-dataset/spect_train.csv
/kaggle/input/spect-heart-dataset/spect_test.csv


In [2]:
from itertools import combinations
from collections import defaultdict

transactions = [
    ['I1', 'I2', 'I5'],
    ['I2', 'I4'],
    ['I2', 'I3'],
    ['I1', 'I2', 'I4'],
    ['I1', 'I3'],
    ['I2', 'I3'],
    ['I1', 'I3'],
    ['I1', 'I2', 'I3', 'I5'],
    ['I1', 'I2', 'I3']
]

def get_items(transactions):
    items = set()
    for transaction in transactions:
        for item in transaction:
            items.add(frozenset([item]))
    return items

def get_support_count(itemset, transactions):
    count = 0
    for transaction in transactions:
        if itemset.issubset(set(transaction)):
            count += 1
    return count

def apriori(transactions, min_support_count):
    frequent_itemsets = {}
    k = 1
    
    current_itemsets = get_items(transactions)
    
    while current_itemsets:
        frequent_k = {}
        for itemset in current_itemsets:
            support_count = get_support_count(itemset, transactions)
            if support_count >= min_support_count:
                frequent_k[itemset] = support_count
        
        if not frequent_k:
            break
            
        frequent_itemsets.update(frequent_k)
        
        items = set()
        for itemset in frequent_k.keys():
            items.update(itemset)
        
        current_itemsets = set()
        for combo in combinations(items, k + 1):
            candidate = frozenset(combo)
            all_subsets_frequent = True
            for subset in combinations(combo, k):
                if frozenset(subset) not in frequent_k:
                    all_subsets_frequent = False
                    break
            if all_subsets_frequent:
                current_itemsets.add(candidate)
        
        k += 1
    
    return frequent_itemsets

def generate_rules(frequent_itemsets, transactions, min_confidence):
    rules = []
    
    for itemset in frequent_itemsets:
        if len(itemset) < 2:
            continue
        
        itemset_support = frequent_itemsets[itemset]
        
        for i in range(1, len(itemset)):
            for antecedent in combinations(itemset, i):
                antecedent = frozenset(antecedent)
                consequent = itemset - antecedent
                
                if antecedent in frequent_itemsets:
                    antecedent_support = frequent_itemsets[antecedent]
                    confidence = (itemset_support / antecedent_support) * 100
                    
                    if confidence >= min_confidence:
                        rules.append({
                            'antecedent': antecedent,
                            'consequent': consequent,
                            'support': itemset_support,
                            'confidence': confidence
                        })
    
    return rules

min_support_count = 2
min_confidence = 70

frequent_itemsets = apriori(transactions, min_support_count)

print("PROBLEM 1: Min Support Count = 2, Min Confidence = 70%")
print("\nFrequent Itemsets:")
for itemset, support in sorted(frequent_itemsets.items(), key=lambda x: (len(x[0]), x[0])):
    print(f"{set(itemset)}: {support}")

rules = generate_rules(frequent_itemsets, transactions, min_confidence)

print("\nStrong Association Rules:")
for rule in rules:
    print(f"{set(rule['antecedent'])} => {set(rule['consequent'])}")
    print(f"  Support: {rule['support']}, Confidence: {rule['confidence']:.2f}%")

PROBLEM 1: Min Support Count = 2, Min Confidence = 70%

Frequent Itemsets:
{'I2'}: 7
{'I5'}: 2
{'I4'}: 2
{'I3'}: 6
{'I1'}: 6
{'I2', 'I3'}: 4
{'I1', 'I5'}: 2
{'I1', 'I2'}: 4
{'I5', 'I2'}: 2
{'I1', 'I3'}: 4
{'I4', 'I2'}: 2
{'I1', 'I2', 'I3'}: 2
{'I1', 'I5', 'I2'}: 2

Strong Association Rules:
{'I5'} => {'I1'}
  Support: 2, Confidence: 100.00%
{'I5'} => {'I2'}
  Support: 2, Confidence: 100.00%
{'I4'} => {'I2'}
  Support: 2, Confidence: 100.00%
{'I5'} => {'I1', 'I2'}
  Support: 2, Confidence: 100.00%
{'I1', 'I5'} => {'I2'}
  Support: 2, Confidence: 100.00%
{'I5', 'I2'} => {'I1'}
  Support: 2, Confidence: 100.00%


In [3]:
from itertools import combinations
from collections import defaultdict

transactions = [
    ['M', 'O', 'N', 'K', 'E', 'Y'],
    ['D', 'O', 'N', 'K', 'E', 'Y'],
    ['M', 'A', 'K', 'E'],
    ['M', 'U', 'C', 'K', 'Y'],
    ['C', 'O', 'O', 'K', 'I', 'E']
]

def get_items(transactions):
    items = set()
    for transaction in transactions:
        for item in transaction:
            items.add(frozenset([item]))
    return items

def get_support_count(itemset, transactions):
    count = 0
    for transaction in transactions:
        if itemset.issubset(set(transaction)):
            count += 1
    return count

def apriori(transactions, min_support_percentage):
    min_support_count = int((min_support_percentage / 100) * len(transactions))
    frequent_itemsets = {}
    k = 1
    
    current_itemsets = get_items(transactions)
    
    while current_itemsets:
        frequent_k = {}
        for itemset in current_itemsets:
            support_count = get_support_count(itemset, transactions)
            if support_count >= min_support_count:
                frequent_k[itemset] = support_count
        
        if not frequent_k:
            break
            
        frequent_itemsets.update(frequent_k)
        
        items = set()
        for itemset in frequent_k.keys():
            items.update(itemset)
        
        current_itemsets = set()
        for combo in combinations(items, k + 1):
            candidate = frozenset(combo)
            all_subsets_frequent = True
            for subset in combinations(combo, k):
                if frozenset(subset) not in frequent_k:
                    all_subsets_frequent = False
                    break
            if all_subsets_frequent:
                current_itemsets.add(candidate)
        
        k += 1
    
    return frequent_itemsets

def generate_rules(frequent_itemsets, transactions, min_confidence):
    rules = []
    
    for itemset in frequent_itemsets:
        if len(itemset) < 2:
            continue
        
        itemset_support = frequent_itemsets[itemset]
        
        for i in range(1, len(itemset)):
            for antecedent in combinations(itemset, i):
                antecedent = frozenset(antecedent)
                consequent = itemset - antecedent
                
                if antecedent in frequent_itemsets:
                    antecedent_support = frequent_itemsets[antecedent]
                    confidence = (itemset_support / antecedent_support) * 100
                    
                    if confidence >= min_confidence:
                        rules.append({
                            'antecedent': antecedent,
                            'consequent': consequent,
                            'support': itemset_support,
                            'confidence': confidence
                        })
    
    return rules

min_support_percentage = 60
min_confidence = 80

frequent_itemsets = apriori(transactions, min_support_percentage)

print("PROBLEM 2: Min Support = 60%, Min Confidence = 80%")
print("\nFrequent Itemsets:")
for itemset, support in sorted(frequent_itemsets.items(), key=lambda x: (len(x[0]), x[0])):
    support_percentage = (support / len(transactions)) * 100
    print(f"{set(itemset)}: {support} ({support_percentage:.1f}%)")

rules = generate_rules(frequent_itemsets, transactions, min_confidence)

print("\nStrong Association Rules:")
for rule in rules:
    print(f"{set(rule['antecedent'])} => {set(rule['consequent'])}")
    print(f"  Support: {rule['support']}, Confidence: {rule['confidence']:.2f}%")

PROBLEM 2: Min Support = 60%, Min Confidence = 80%

Frequent Itemsets:
{'O'}: 3 (60.0%)
{'E'}: 4 (80.0%)
{'K'}: 5 (100.0%)
{'M'}: 3 (60.0%)
{'Y'}: 3 (60.0%)
{'K', 'M'}: 3 (60.0%)
{'O', 'E'}: 3 (60.0%)
{'O', 'K'}: 3 (60.0%)
{'E', 'K'}: 4 (80.0%)
{'K', 'Y'}: 3 (60.0%)
{'O', 'K', 'E'}: 3 (60.0%)

Strong Association Rules:
{'M'} => {'K'}
  Support: 3, Confidence: 100.00%
{'O'} => {'E'}
  Support: 3, Confidence: 100.00%
{'O'} => {'K'}
  Support: 3, Confidence: 100.00%
{'E'} => {'K'}
  Support: 4, Confidence: 100.00%
{'K'} => {'E'}
  Support: 4, Confidence: 80.00%
{'Y'} => {'K'}
  Support: 3, Confidence: 100.00%
{'O'} => {'E', 'K'}
  Support: 3, Confidence: 100.00%
{'O', 'K'} => {'E'}
  Support: 3, Confidence: 100.00%
{'O', 'E'} => {'K'}
  Support: 3, Confidence: 100.00%


In [4]:
from itertools import combinations
import pandas as pd

csv_path = "/kaggle/input/spect-heart-dataset/spect_train.csv"

df = pd.read_csv(csv_path, header=None)
df = df.iloc[:20, :5]

transactions = []
for _, row in df.iterrows():
    t = []
    for i, v in enumerate(row):
        t.append(f"F{i}_{v}")
    transactions.append(t)

def get_items(transactions):
    items = set()
    for t in transactions:
        for i in t:
            items.add(frozenset([i]))
    return items

def support_count(itemset, transactions):
    c = 0
    for t in transactions:
        if itemset.issubset(set(t)):
            c += 1
    return c

def apriori(transactions, min_support_pct):
    min_support = int((min_support_pct / 100) * len(transactions))
    freq = {}
    k = 1
    current = get_items(transactions)

    while current:
        freq_k = {}
        for itemset in current:
            sc = support_count(itemset, transactions)
            if sc >= min_support:
                freq_k[itemset] = sc

        if not freq_k:
            break

        freq.update(freq_k)

        items = set()
        for fs in freq_k:
            items |= fs

        current = set()
        for combo in combinations(items, k + 1):
            c = frozenset(combo)
            ok = True
            for sub in combinations(combo, k):
                if frozenset(sub) not in freq_k:
                    ok = False
                    break
            if ok:
                current.add(c)

        k += 1

    return freq

def generate_rules(freq, min_conf):
    rules = []
    for itemset in freq:
        if len(itemset) < 2:
            continue
        for i in range(1, len(itemset)):
            for ant in combinations(itemset, i):
                ant = frozenset(ant)
                cons = itemset - ant
                conf = (freq[itemset] / freq[ant]) * 100
                if conf >= min_conf:
                    rules.append((ant, cons, freq[itemset], conf))
    return rules

min_support = 60
min_confidence = 80

freq_itemsets = apriori(transactions, min_support)
rules = generate_rules(freq_itemsets, min_confidence)

print("Frequent Itemsets")
for fs, sc in sorted(freq_itemsets.items(), key=lambda x: (len(x[0]), x[0])):
    print(set(fs), sc)

print("\nStrong Association Rules")
for a, c, s, conf in rules:
    print(set(a), "=>", set(c), "Support:", s, "Confidence:", conf)


Frequent Itemsets
{'F0_1'} 19

Strong Association Rules
